## Sel 1: Impor Pustaka dan Fungsi Parser Angka

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import plotly.express as px

def parse_indo_number(val):
    """
    Fungsi parser tunggal: Mengubah format angka Indonesia menjadi float.
    Tanda '-' dan nilai kosong akan dikonversi menjadi 0.0.
    """
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip()
    if val_str in ['-', '‒', '']:
        return 0.0
    # Hilangkan pemisah ribuan (titik) dan ubah koma desimal menjadi titik
    val_str = val_str.replace('.', '').replace(',', '.')
    try:
        return float(val_str)
    except ValueError:
        return 0.0

def format_indo_number(val):
    """Pembantu visual: Mencetak angka float kembali ke teks format Indonesia."""
    if pd.isna(val) or val is None:
        return "-"
    return f"{val:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')

## Sel 2: Membaca, Membersihkan, dan Mengagregasi Data

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import gdown

# 2. Unduh dan Muat Data dari Google Drive menggunakan File ID
FILE_ID = "1nMGDNNhcpG1OANTmLviTFmi0mRZo9yTY"
url = f'https://docs.google.com/spreadsheets/d/{FILE_ID}/export?format=xlsx'
output_filename = "data_pangan.xlsx"

# Mengunduh file ke lingkungan Colab
gdown.download(url, output_filename, quiet=False)

# Membaca data
df = pd.read_excel(output_filename, sheet_name="Sheet3", header=1)

nodes = []
current_group = None
root_id = "Makanan"

total_makanan = 0.0
group_children = {}
MIN_QTY = 0.10
null_reasons = {'qty_too_low_or_zero': 0, 'mixed_or_null_units_in_group': 0}

for idx, row in df.iterrows():
    no_val = str(row['No.']).strip()
    jenis = str(row['Jenis Makanan']).strip()
    satuan = str(row['Satuan']).strip() if pd.notna(row['Satuan']) else None

    qty = parse_indo_number(row['Banyaknya'])
    nilai = parse_indo_number(row['Nilai (Rp)'])

    if 'jumlah' in jenis.lower() or 'total' in jenis.lower():
        continue

    # Deteksi KELOMPOK (Level 2)
    if re.match(r'^[A-Z]\.$', no_val):
        current_group = {
            'id': f"{root_id}/{jenis}",
            'parent': root_id,
            'label': jenis,
            'level': 2,
            'pengeluaran': 0.0,
            'kuantitas': None, # PERBAIKAN: Default null, bukan 0.0
            'satuan': set(),
            'children': [],
            'all_children_have_price': True
        }
        nodes.append(current_group)
        group_children[current_group['id']] = current_group

    # Deteksi KOMODITAS (Level 3)
    elif current_group is not None and no_val.isdigit():
        comm_id = f"{current_group['id']}/{jenis}"

        if qty >= MIN_QTY:
            harga = nilai / qty
            has_price = True
        else:
            harga = None
            has_price = False
            null_reasons['qty_too_low_or_zero'] += 1

        comm = {
            'id': comm_id,
            'parent': current_group['id'],
            'label': jenis,
            'level': 3,
            'pengeluaran': nilai,
            'kuantitas': qty,
            'satuan': satuan,
            'harga_implisit': harga
        }
        nodes.append(comm)

        current_group['pengeluaran'] += nilai
        if satuan and satuan != 'nan':
            current_group['satuan'].add(satuan)
        current_group['children'].append(comm)
        if not has_price:
            current_group['all_children_have_price'] = False

        total_makanan += nilai

## Sel 3: Pembangunan Pohon Hierarki, Metrik Share, dan Ekspor JSON

In [ ]:
# Membangun node akar (Root) dengan pembulatan 2 desimal
root_node = {
    'id': root_id,
    'parent': "",
    'label': "Total Makanan",
    'level': 1,
    'pengeluaran': round(total_makanan, 2),
    'kuantitas': None,
    'satuan': None,
    'harga_implisit': None,
    'share_dari_total': 100.0,
    'share_dari_induk': 100.0
}
final_nodes = [root_node]
empty_groups = []

for node in nodes:
    # PERBAIKAN: Bulatkan nilai pengeluaran induk sebelum diolah lebih lanjut
    node['pengeluaran'] = round(node['pengeluaran'], 2)

    if node['level'] == 2:
        if not node['children']:
            node['level'] = 3
            node['satuan'] = None
            node['harga_implisit'] = None
            node['kuantitas'] = None
            empty_groups.append(node['label'])
        else:
            if node['all_children_have_price'] and len(node['satuan']) == 1:
                node['satuan'] = list(node['satuan'])[0]
                total_qty = sum(c['kuantitas'] for c in node['children'])
                node['kuantitas'] = round(total_qty, 2)
                node['harga_implisit'] = round(node['pengeluaran'] / node['kuantitas'], 2) if node['kuantitas'] > 0 else None
            else:
                node['satuan'] = None
                node['harga_implisit'] = None
                node['kuantitas'] = None # Pastikan tetap null jika satuan bervariasi
                null_reasons['mixed_or_null_units_in_group'] += 1

        # PERBAIKAN: Pembulatan Share 2 desimal
        node['share_dari_total'] = round((node['pengeluaran'] / total_makanan) * 100, 2) if total_makanan else 0
        node['share_dari_induk'] = node['share_dari_total']

        node.pop('children', None)
        node.pop('all_children_have_price', None)
        if isinstance(node['satuan'], set):
            node['satuan'] = None

        final_nodes.append(node)

    elif node['level'] == 3:
        # PERBAIKAN: Pembulatan Share dan Harga 2 desimal
        node['share_dari_total'] = round((node['pengeluaran'] / total_makanan) * 100, 2) if total_makanan else 0

        parent_pengeluaran = group_children[node['parent']]['pengeluaran']
        node['share_dari_induk'] = round((node['pengeluaran'] / parent_pengeluaran) * 100, 2) if parent_pengeluaran else 0

        if node['harga_implisit'] is not None:
            node['harga_implisit'] = round(node['harga_implisit'], 2)

        final_nodes.append(node)

print(f"Total Nilai: Rp {format_indo_number(total_makanan)}")
print(f"Node Null karena qty < {MIN_QTY}: {null_reasons['qty_too_low_or_zero']}")
print(f"Kelompok Null karena multivariasi satuan: {null_reasons['mixed_or_null_units_in_group']}")

with open('hierarki_flat.json', 'w') as f:
    json.dump(final_nodes, f, indent=4)
print("File JSON diekspor ke 'hierarki_flat.json'")

## Sel 4: Analisis Eksplorasi Data (EDA)

In [ ]:
# Ekstrak data untuk EDA (H6)
groups = [n for n in final_nodes if n['level'] == 2]
comms = [n for n in final_nodes if n['level'] == 3]

beras = next((n for n in comms if n['label'] == 'Beras'), None)
padi_padian = next((n for n in groups if n['label'] == 'PADI-PADIAN'), None)
top_3_groups = [g['label'] for g in sorted(groups, key=lambda x: x['pengeluaran'], reverse=True)[:3]]
top_5_comms = [c['label'] for c in sorted(comms, key=lambda x: x['pengeluaran'], reverse=True)[:5]]

protein_groups = ['IKAN/UDANG/CUMI/KERANG', 'DAGING', 'TELUR DAN SUSU', 'KACANG-KACANGAN']
prep_tobacco = ['MAKANAN DAN MINUMAN JADI', 'ROKOK']
protein_share = sum(g['share_dari_total'] for g in groups if g['label'] in protein_groups)
prep_tobacco_share = sum(g['share_dari_total'] for g in groups if g['label'] in prep_tobacco)

kg_comms = [n for n in comms if n['satuan'] == 'kg' and n['harga_implisit'] is not None]
kg_sorted = sorted(kg_comms, key=lambda x: x['harga_implisit'])

print("=== INSIGHT EDA ===")
print(f"1. Beras menyumbang {format_indo_number(beras['share_dari_total'])}% dari total makanan dan memonopoli kelompoknya dengan pangsa {format_indo_number(beras['share_dari_induk'])}%.")
print(f"2. Tiga kelompok makanan terbesar adalah {', '.join(top_3_groups)}.")
print(f"3. Lima komoditas dengan nilai tertinggi mencakup {', '.join(top_5_comms)}.")
print(f"4. Pangsa Makanan/Minuman Jadi dan Rokok sangat dominan mencapai {format_indo_number(prep_tobacco_share)}%, hampir tiga kali lipat gabungan seluruh kelompok protein ({format_indo_number(protein_share)}%).")
print(f"5. Di antara komoditas (satuan kg) yang memenuhi syarat hitung, {kg_sorted[-1]['label']} merupakan yang termahal (Rp {format_indo_number(kg_sorted[-1]['harga_implisit'])}/kg) dan {kg_sorted[0]['label']} merupakan yang termurah (Rp {format_indo_number(kg_sorted[0]['harga_implisit'])}/kg).")

## Sel 5: Prototipe Visualisasi Plotly untuk React

In [ ]:
import plotly.express as px
import pandas as pd
import json
import plotly.io as pio

# 1. Memuat Data JSON
with open('hierarki_flat.json', 'r') as f:
    plot_data = json.load(f)

# ==========================================
# FILTER STRATEGI: ISI KOTAK PENUH (TANPA RUANG KOSONG)
# ==========================================
SEKTOR_TARGET = ['PADI-PADIAN', 'SAYUR-SAYURAN', 'TELUR DAN SUSU', 'ROKOK', 'MAKANAN DAN MINUMAN JADI']

kelompok_semua = [n for n in plot_data if n['level'] == 2]
komoditas_semua = [n for n in plot_data if n['level'] == 3]

processed_nodes = []
total_akar_baru = 0.0

for kel in kelompok_semua:
    if kel['label'] in SEKTOR_TARGET:
        # Ambil anak komoditas di sektor ini, urutkan, dan ambil Top 5 saja
        anak_kel = [k for k in komoditas_semua if k['parent'] == kel['id']]
        top_5_anak = sorted(anak_kel, key=lambda x: x['pengeluaran'], reverse=True)[:5]

        # TRIK UTAMA: Hitung total pengeluaran murni dari 5 komoditas teratas ini saja
        sum_top5 = sum(anak['pengeluaran'] for anak in top_5_anak)

        # Timpa nilai asli kelompok dengan nilai Top 5 agar kotak 100% penuh tanpa sisa
        kel['pengeluaran'] = sum_top5
        if kel.get('satuan') is None or str(kel.get('satuan')).lower() in ['none', 'nan']:
            kel['satuan'] = 'Bervariasi'

        processed_nodes.append(kel)

        # Masukkan ke-5 komoditas tersebut
        for anak in top_5_anak:
            if anak.get('satuan') is None or str(anak.get('satuan')).lower() == 'nan':
                anak['satuan'] = '-'
            processed_nodes.append(anak)

        total_akar_baru += sum_top5

# Buat Node Akar (Total) baru yang nilainya menyesuaikan penjumlahan kotak
akar = {
    'id': 'Makanan', 'parent': '', 'label': 'Total Makanan Terpilih', 'level': 1,
    'pengeluaran': total_akar_baru, 'kuantitas': None, 'satuan': 'Bervariasi',
    'harga_implisit': None
}
processed_nodes.insert(0, akar)

df_plot = pd.DataFrame(processed_nodes)

# Imputasi warna pada kelompok agar gradasi aman
median_val = df_plot['harga_implisit'].median()
df_plot['harga_tampilan'] = df_plot['harga_implisit'].fillna(median_val)
df_plot['hover_harga'] = df_plot['harga_implisit'].apply(
    lambda x: f"Rp {x:,.2f} / satuan" if pd.notna(x) else "Tidak Dapat Dihitung"
)

# ==========================================
# SKALA WARNA: BIRU TEDUH (TIDAK SILAU)
# ==========================================
# Palet biru tanpa warna putih pucat: Sky Blue -> Dodger Blue -> Royal Blue -> Medium Blue -> Navy
warna_biru_teduh = ["#90CAF9", "#64B5F6", "#42A5F5", "#2196F3", "#1976D2", "#1565C0", "#0D47A1"]

# Memotong batas ekstrem agar gradasi lebih merata
batas_bawah = df_plot['harga_implisit'].quantile(0.10)
batas_atas = df_plot['harga_implisit'].quantile(0.90)


# ==========================================
# 2. VISUALISASI TREEMAP (KOTAK 100% PENUH)
# ==========================================
fig_tree = px.treemap(
    df_plot,
    ids='id', names='label', parents='parent', values='pengeluaran',
    color='harga_tampilan',
    color_continuous_scale=warna_biru_teduh,
    range_color=[batas_bawah, batas_atas],
    branchvalues='total'
)

fig_tree.update_traces(
    textinfo="label+value", # Tampilkan nama + rupiah
    textfont=dict(color='white', size=13),
    marker=dict(
        cornerradius=0,
        showscale=False, # Sembunyikan legenda agar rapi
        line=dict(color='#FFFFFF', width=2)
    ),
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Pengeluaran: Rp %{value:,.2f}<br>"
        "Satuan: %{customdata[0]}<br>"
        "Harga Implisit: %{customdata[1]}<br>"
        "<extra></extra>"
    ),
    customdata=df_plot[['satuan', 'hover_harga']]
)
fig_tree.update_layout(margin=dict(t=10, l=10, r=10, b=10), paper_bgcolor='white')
pio.show(fig_tree)
fig_tree.write_html("treemap_makalah_final.html")


# ==========================================
# 3. VISUALISASI SUNBURST
# ==========================================
fig_sun = px.sunburst(
    df_plot,
    ids='id', names='label', parents='parent', values='pengeluaran',
    color='harga_tampilan',
    color_continuous_scale=warna_biru_teduh,
    range_color=[batas_bawah, batas_atas],
    branchvalues='total'
)

fig_sun.update_traces(
    textinfo="label",
    textfont=dict(color='white', size=12),
    insidetextorientation='radial',
    marker=dict(
        showscale=False,
        line=dict(color='#FFFFFF', width=1.5)
    ),
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Pengeluaran: Rp %{value:,.2f}<br>"
        "Satuan: %{customdata[0]}<br>"
        "Harga Implisit: %{customdata[1]}<br>"
        "<extra></extra>"
    ),
    customdata=df_plot[['satuan', 'hover_harga']]
)
fig_sun.update_layout(margin=dict(t=10, l=10, r=10, b=10), paper_bgcolor='white')
pio.show(fig_sun)
fig_sun.write_html("sunburst_makalah_final.html")

# ==========================================
# 4. EKSPOR JSON KHUSUS UNTUK REACT
# ==========================================
# Menyimpan data yang sudah disaring & dihitung ulang ke JSON baru
with open('hierarki_react_final.json', 'w') as f:
    json.dump(processed_nodes, f, indent=4)

print("✅ File JSON khusus React ('hierarki_react_final.json') berhasil diekspor!")